# Extra B · Data storytelling with Streamlit

**Objective:** To learn how to build and share interactive web applications from Python scripts using Streamlit.

## Part 1: Concepts (90 mins)

### 1. What is Streamlit?
Streamlit is an open-source Python library that makes it easy to create and share beautiful, custom web apps for machine learning and data science. 

**Core Principles:**
- **Embrace Scripting:** A Streamlit app is just a Python script. There's no hidden state.
- **Interactive Widgets:** Add widgets like sliders, buttons, and dropdowns with a single line of code.
- **Instant Deployment:** Easily share your apps with others.

### 2. How it Works
When you run a Streamlit script, it starts a local web server. Every time you interact with a widget, the script is re-run from top to bottom, and the web page is updated with the results. Streamlit's caching mechanism ensures that long-running computations are not re-executed unnecessarily.

### 3. Core Components
You can display almost anything in Streamlit:
- `st.title()`, `st.header()`, `st.write()` for text and data.
- `st.dataframe()` to display pandas DataFrames.
- `st.pyplot()` or `st.plotly_chart()` for plots.
- `st.slider()`, `st.selectbox()`, `st.button()` for user input.

### 4. Writing a Streamlit App

**Important Note:** Streamlit apps are Python scripts (`.py`), not Jupyter Notebooks. We will use a special command `%%writefile` to create the script from this notebook. You will then run it from your terminal.

In [ ]:
%%writefile app.py
import streamlit as st
import pandas as pd
import matplotlib.pyplot as plt

# --- Page Configuration ---
st.set_page_config(
    page_title="Sales Data Dashboard",
    page_icon="📊",
    layout="wide"
)

# --- Data Loading ---
# Use a function with caching for performance
@st.cache_data
def load_data():
    df = pd.read_csv('messy_sales_data.csv')
    # Basic cleanup
    df = df.drop_duplicates()
    df['Quantity'] = df['Quantity'].fillna(df['Quantity'].median())
    df['Region'] = df['Region'].fillna('Unknown').str.strip().str.title()
    return df

df = load_data()

# --- App Layout ---
st.title("📊 Interactive Sales Dashboard")

# --- Sidebar for Filters ---
st.sidebar.header("Filters")
selected_region = st.sidebar.selectbox(
    'Select a Region:',
    options=['All'] + sorted(df['Region'].unique())
)

price_range = st.sidebar.slider(
    'Select a Price Range:',
    min_value=float(df['Price'].min()),
    max_value=float(df['Price'].max()),
    value=(float(df['Price'].min()), float(df['Price'].max()))
)

# --- Filtering Data ---
if selected_region != 'All':
    df_filtered = df[df['Region'] == selected_region]
else:
    df_filtered = df.copy()

df_filtered = df_filtered[
    (df_filtered['Price'] >= price_range[0]) &
    (df_filtered['Price'] <= price_range[1])
]

# --- Main Panel ---
st.header("Filtered Data")
st.dataframe(df_filtered)

st.header("Data Visualizations")

# Create two columns for charts
col1, col2 = st.columns(2)

with col1:
    st.subheader("Sales by Region")
    sales_by_region = df_filtered.groupby('Region')['Price'].sum()
    fig1, ax1 = plt.subplots()
    sales_by_region.plot(kind='bar', ax=ax1, color='skyblue')
    ax1.set_ylabel("Total Sales")
    ax1.set_xlabel("Region")
    plt.xticks(rotation=45)
    st.pyplot(fig1)

with col2:
    st.subheader("Quantity Distribution")
    fig2, ax2 = plt.subplots()
    df_filtered['Quantity'].plot(kind='hist', bins=20, ax=ax2, color='salmon')
    ax2.set_xlabel("Quantity")
    st.pyplot(fig2)


## Part 2: How to Run Your App

1. **Save the code:** The cell above created a file named `app.py` in the same directory as this notebook.

2. **Open your terminal or command prompt.**

3. **Make sure you have streamlit installed:**
   ```bash
   pip install streamlit
   ```

4. **Navigate to the correct directory** where you saved `app.py`.

5. **Run the app with the following command:**
   ```bash
   streamlit run app.py
   ```

A new tab should open in your web browser with your interactive dashboard!